# Data transcript analysis

<a href="https://colab.research.google.com/github/PeaceAndLongLife/Analysis-Colab/blob/Development/notebooks/Data_used.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Load Git Repository
# @markdown Clone the Git repository for analysis if needed.
clone_git_repo = False  # @param {type:"boolean"}
if clone_git_repo:
    !git clone https://github.com/PeaceAndLongLife/Analysis-Colab.git
    %cd Analysis-Colab
    
    import sys
    from pathlib import Path
    sys.path.append(str(Path("src").resolve()))
    %cd notebooks


## Setup Google API

In [ ]:
# @title Mount google drive and read in the SERVICE_ACCOUNT_FILE  {"form-width":"20%"}

# @markdown ---
# @markdown
# @markdown The `SERVICE_ACCOUNT_FILE` path is stored as a secret in google colab. If you do not have this sotred on your colab, contact the Admin: Travis Kregear at tkregear@pdx.edu

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# load SERVICE_ACCOUNT_FIL
from google.colab import userdata

SERVICE_ACCOUNT_FILE = userdata.get('SERVICE_ACCOUNT_FILE')

# @title ## Install pyDrive  {"form-width":"20%"}

# @markdown ---
# @markdown Installing PyDrive
# @markdown
# @markdown Thie is necessary if using the Google API to call files by their file_id

# !pip install PyDrive

import sys
# 2. Tell Python to look in your custom folder
# sys.path.append(PACKAGE_PATH)
sys.path.append('src')
# 3. Import your file!
from GoogleFunctions import extract_file_id
from local_io import read_csv_from_id


## Read in data


In [ ]:
# @title Data file info {"form-width":"20%"}

# UserProfile Info
# @markdown Read in userprofile file 
userprofile_file_link = "" # @param {"type":"string"}
userprofile_file_link_id = extract_file_id(userprofile_file_link)
show_userprofile = False # @param {"type":"boolean"}

# Consent data info
# @markdown Read in consent file 

consent_file_link = "" # @param {"type":"string"}
consent_file_link_id = extract_file_id(consent_file_link)
show_consent = False # @param {"type":"boolean"}

# Transcript Data info
# @markdown Read in transcript file 
trans_file_link = "" # @param {"type":"string"}
trans_file_link_id = extract_file_id(trans_file_link)
show_trans = False # @param {"type":"boolean"}
show_remove_staff_data = True # @param {"type":"boolean"}

userprofile_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, userprofile_file_link_id, show_userprofile)
consent_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, consent_file_link_id, show_consent)
trans_df = read_csv_from_id(SERVICE_ACCOUNT_FILE, trans_file_link_id, show_trans)

from data_scrub import remove_staff, process_messages, parse_messages
# import json

if show_remove_staff_data:
    userprofile_df, trans_df = remove_staff(
        userprofile_df,
        trans_df,
        [
          'travis@pdx.edu',
          'marie.snetinova@matfyz.cuni.cz',
          'mptl_group_1@pdx.edu',
          'mptl_group_2@pdx.edu',
          'mptl_group_3@pdx.edu',
          'mptl_group_4@pdx.edu',
          'mptl_group_5@pdx.edu',
          'mptl_group_6@pdx.edu',
          'mptl_group_1@pdx.edu',
          'mptl_user_1@pdx.edu',
          'mptl_user_10@pdx.edu'
        ])

# Merge consent into transcript
combined_df = process_messages(consent_df, trans_df)


# Apply the parsing function to the 'all_messages' column
combined_df['all_messages'] = combined_df['all_messages'].apply(parse_messages)

# Count message objects in 'all_messages' and add to 'interactions' column
combined_df['interactions'] = combined_df['all_messages'].apply(len)

combined_df = combined_df[combined_df['all_messages'].apply(lambda x: len(x) > 0)]

print("Combined DataFrame with split question columns and interactions:")
# display(combined_df.head())

In [ ]:
# @title analyze_stats function
def analyze_hist(df, drop_counts_under_param,y,x):
  # Exclude users with less than drop_counts_under_param unique threads
  df_filtered = df[df[y] >= drop_counts_under_param]


  ### Plot distribution
  import matplotlib.pyplot as plt
  import seaborn as sns

  # Set the style for the plot
  sns.set_style("whitegrid")

  plt.figure(figsize=(10, 6))
  sns.histplot(df_filtered[y], bins=5, kde=True)
  plt.title(f"Distribution of {y} per {x}")
  plt.xlabel(f"{x}")
  # plt.ylabel(f"Number of {y}")
  plt.tight_layout()
  plt.show()
  display(df_filtered[y].describe())


In [ ]:
rating_df = combined_df[combined_df['data']==True]
